# 01 Lead History Clean

This notebook only does base cleaning plus duplicate review.
It does **not** assign final subsystem ids and does **not** merge with history.

Enter the file name in the next cell. The workbook must be in the same folder as this notebook.


In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

BASE_DIR = Path('.').resolve()
CLEAN_OUTPUT_DIR = BASE_DIR / 'clean_pipeline'
CLEAN_OUTPUT_DIR.mkdir(exist_ok=True)

new_workbook_name = input(
    'Enter the workbook file name in the current Pipeline folder '
    '(example: Public_Water_Supply_90th_Percentiles_2025.xlsx): '
).strip()

NEW_WORKBOOK_PATH = BASE_DIR / new_workbook_name

if not NEW_WORKBOOK_PATH.exists():
    raise FileNotFoundError(f'File not found: {NEW_WORKBOOK_PATH}')

print('Workbook to clean:', NEW_WORKBOOK_PATH)
print('Output folder:', CLEAN_OUTPUT_DIR)

Workbook to clean: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\Public_Water_Supply_90th_Percentiles_2025.xlsx
Output folder: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\clean_pipeline


In [3]:
STANDARD_COLUMNS = [
    'pwsid',
    'system_name',
    'county',
    'population',
    'monitoring_end_date',
    'lead_90th_ppb',
    'includes_5th_liter_or_not',
    'sampling_next_due_subject_to_change',
    'year',
    'month',
    'above_action_level',
]

RAW_TO_STANDARD = {
    'public_water_supply_id': 'pwsid',
    'system_name': 'system_name',
    'county': 'county',
    'population': 'population',
    'last_monitoring_period_end': 'monitoring_end_date',
    'lead_90th_percentile_ppb': 'lead_90th_ppb',
    'includes_5th_liter?': 'includes_5th_liter_or_not',
    'sampling_next_due_subject_to_change': 'sampling_next_due_subject_to_change',
}


def normalize_header(value):
    return (
        str(value)
        .replace('\n', ' ')
        .strip()
        .lower()
        .replace(' ', '_')
        .replace('(', '')
        .replace(')', '')
    )


def find_history_sheet(workbook_path):
    excel = pd.ExcelFile(workbook_path)

    for sheet_name in excel.sheet_names:
        preview = pd.read_excel(workbook_path, sheet_name=sheet_name, nrows=5)
        preview = preview.dropna(axis=1, how='all')
        if preview.shape[1] < 8:
            continue

        first_eight = [normalize_header(col) for col in preview.columns[:8]]
        if 'public_water_supply_id' in first_eight and 'lead_90th_percentile_ppb' in first_eight:
            return sheet_name

    return excel.sheet_names[0]


def clean_history_workbook(workbook_path):
    history_data = pd.read_excel(workbook_path, sheet_name=find_history_sheet(workbook_path))
    history_data = history_data.dropna(axis=1, how='all')
    history_data = history_data.iloc[:, 0:8].copy()
    history_data.columns = [normalize_header(col) for col in history_data.columns]
    history_data = history_data.rename(columns=RAW_TO_STANDARD)

    missing = [col for col in RAW_TO_STANDARD.values() if col not in history_data.columns]
    if missing:
        raise ValueError(f'Missing expected columns in workbook: {missing}')

    history_data['monitoring_end_date'] = pd.to_datetime(history_data['monitoring_end_date'], errors='coerce')
    history_data['sampling_next_due_subject_to_change'] = pd.to_datetime(
        history_data['sampling_next_due_subject_to_change'],
        errors='coerce',
    )
    history_data['population'] = pd.to_numeric(history_data['population'], errors='coerce')
    history_data['lead_90th_ppb'] = pd.to_numeric(history_data['lead_90th_ppb'], errors='coerce')

    history_data = history_data.dropna(
        subset=['pwsid', 'monitoring_end_date', 'lead_90th_ppb']
    ).copy()

    for col in ['pwsid', 'system_name', 'county', 'includes_5th_liter_or_not']:
        history_data[col] = history_data[col].astype(str).str.strip()

    history_data['year'] = history_data['monitoring_end_date'].dt.year
    history_data['month'] = history_data['monitoring_end_date'].dt.month
    history_data['above_action_level'] = history_data['lead_90th_ppb'] > 12

    history_data = history_data[STANDARD_COLUMNS].copy()
    history_data = history_data.drop_duplicates().reset_index(drop=True)

    duplicate_checking = history_data.copy()
    duplicate_checking['lead_value_count'] = (
        duplicate_checking
        .groupby(['pwsid', 'monitoring_end_date'])['lead_90th_ppb']
        .transform('nunique')
    )
    duplicate_checking['system_name_count'] = (
        duplicate_checking
        .groupby(['pwsid', 'monitoring_end_date'])['system_name']
        .transform('nunique')
    )

    need_review = duplicate_checking[
        duplicate_checking['lead_value_count'] > 1
    ].sort_values(['pwsid', 'monitoring_end_date', 'system_name']).reset_index(drop=True)

    return history_data, duplicate_checking, need_review


In [5]:
cleaned_history, duplicate_checking, need_review = clean_history_workbook(NEW_WORKBOOK_PATH)

summary_df = pd.DataFrame([
    {
        'rows_after_base_clean': len(cleaned_history),
        'need_review_rows': len(need_review),
        'exact_duplicate_rows_remaining': int(cleaned_history.duplicated().sum()),
    }
])

display(summary_df)
display(cleaned_history.head())
display(need_review.head())

print('After reviewing need_review.csv, save a manually checked file named like:')
print(Path(new_workbook_name).stem + '_cleaned_reviewed.csv')

,rows_after_base_clean,need_review_rows,exact_duplicate_rows_remaining
0,1386,9,0


,pwsid,system_name,county,population,monitoring_end_date,lead_90th_ppb,includes_5th_liter_or_not,sampling_next_due_subject_to_change,year,month,above_action_level
0,MI0000011,ACME TOWNSHIP - HOPE VILLAGE,GRAND TRAVERSE,128,2023-12-31,0,N,2026-09-30,2023,12,False
1,MI0062841,MEDILODGE OF STERLING,ARENAC,39,2025-12-31,888,N,2026-06-30,2025,12,True
2,MI0040455,WOODS AND FIELDS COMMUNITIES WEST,SHIAWASSEE,185,2025-12-31,135,N,2026-06-30,2025,12,True
3,MI0000030,ADDISON,LENAWEE,605,2023-12-31,2,N,2026-09-30,2023,12,False
4,MI0000040,ADRIAN,LENAWEE,23663,2023-12-31,0,N,2026-09-30,2023,12,False


,pwsid,system_name,county,population,monitoring_end_date,lead_90th_ppb,includes_5th_liter_or_not,sampling_next_due_subject_to_change,year,month,above_action_level,lead_value_count,system_name_count
0,MI0000347,BAIR LAKE BIBLE CAMP (DS002),CASS,372,2025-12-31,0,N,2028-09-30,2025,12,False,3,5
1,MI0000347,BAIR LAKE BIBLE CAMP (DS003),CASS,372,2025-12-31,2,N,2026-09-30,2025,12,False,3,5
2,MI0000347,BAIR LAKE BIBLE CAMP (DS004),CASS,372,2025-12-31,1,N,2028-09-30,2025,12,False,3,5
3,MI0000347,BAIR LAKE BIBLE CAMP (DS005),CASS,372,2025-12-31,0,N,2026-09-30,2025,12,False,3,5
4,MI0000347,BAIR LAKE BIBLE CAMP (DS006),CASS,372,2025-12-31,0,N,2028-09-30,2025,12,False,3,5


After reviewing need_review.csv, save a manually checked file named like:
Public_Water_Supply_90th_Percentiles_2025_cleaned_reviewed.csv


In [6]:
output_stem = Path(new_workbook_name).stem

cleaned_path = CLEAN_OUTPUT_DIR / f'{output_stem}_cleaned.csv'
need_review_path = CLEAN_OUTPUT_DIR / f'{output_stem}_need_review.csv'

cleaned_history.to_csv(cleaned_path, index=False, date_format='%Y-%m-%d')
need_review.to_csv(need_review_path, index=False, date_format='%Y-%m-%d')

print('Saved:', cleaned_path)
print('Saved:', need_review_path)
print('Next manual step: review need_review.csv, then save a reviewed copy of cleaned.csv for notebook 02.')

Saved: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\clean_pipeline\Public_Water_Supply_90th_Percentiles_2025_cleaned.csv
Saved: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\clean_pipeline\Public_Water_Supply_90th_Percentiles_2025_need_review.csv
Next manual step: review need_review.csv, then save a reviewed copy of cleaned.csv for notebook 02.
